### Chequear por errores en lo provisto por Gemini

### Nixtla

In [ ]:
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.losses.pytorch import MAE, MQLoss

# ==========================================
# 1. LOAD & PREPARE THE DATA
# ==========================================
# Load your challenge data
X_train = pd.read_csv("./Data/SNCF/hard/Xtrain_hgcGIrA.csv")
Y_train = pd.read_csv("./Data/SNCF/hard/Ytrain_yL5OjS4.csv")
X_test = pd.read_csv("./Data/SNCF/hard/Xtest.csv")

# Combine target with features for Nixtla formatting
train_df = X_train.copy()
train_df["y"] = Y_train["p0q0"]  # Assuming target column is named 'target'

# Construct the mandatory unique_id (Group by Train + Station sequence)
# In this challenge, the unique time-series entity can be defined by the specific Train run on a Way
train_df["unique_id"] = (
    train_df["train"].astype(str) + "_" + train_df["hour"].astype(str)
)
X_test["unique_id"] = X_test["train"].astype(str) + "_" + X_test["hour"].astype(str)

# Create an incremental pseudo-datestamp 'ds' out of the date and hour
# because Nixtla relies on chronological sequencing.
train_df["ds"] = pd.to_datetime(train_df["date"].astype(str) + " " + train_df["hour"], format='mixed')
X_test["ds"] = pd.to_datetime(X_test["date"].astype(str) + " " + X_test["hour"], format='mixed')

# Sort values to prevent sequence disruption
train_df = train_df.sort_values(by=["unique_id", "ds"]).reset_index(drop=True)
X_test = X_test.sort_values(by=["unique_id", "ds"]).reset_index(drop=True)

# Define feature arrays based on the challenge description
# 6 Lags are past covariates; hour and composition can act as known context
hist_exog_cols = ["p1q0", "p2q0", "p3q0", "p0q1", "p0q2", "p0q3"]
stat_exog_cols = ["composition", "way"]

# Handle NAs: Nixtla's TFT handles metadata well, but requires NaN padding for inputs
train_df[hist_exog_cols] = train_df[hist_exog_cols].fillna(0.0)
train_df['hour']=train_df['hour'].fillna(0)
X_test[hist_exog_cols] = X_test[hist_exog_cols].fillna(0.0)

# ==========================================
# 2. INSTANTIATE THE NIXTLA TFT MODEL
# ==========================================
# Horizon is 1 because the goal is "to predict the occupancy rate at the next station only"
horizon = 1

# We look back at least 3 steps (matching our max lag p3/q3)
input_size = 3

tft_model = TFT(
    h=horizon,
    input_size=input_size,
    loss=MQLoss(
        quantiles=[0.1, 0.5, 0.9]
    ),  # Quantile loss maps beautifully to crowding risks
    hist_exog_list=hist_exog_cols,
    stat_exog_list=stat_exog_cols,
    scaler_type="robust",  # Scales occupancy ratios natively [0, 1]
    max_steps=500,  # Number of training iterations
    learning_rate=1e-3,
    batch_size=64,
)

# Initialize the NeuralForecast wrapper
nf = NeuralForecast(models=[tft_model], freq="H")  # 'H' for Hourly scale matching data

# ==========================================
# 3. TRAIN & PREDICT
# ==========================================
print("Training Temporal Fusion Transformer via Nixtla...")
nf.fit(df=train_df)

print("Generating predictions for next-station occupancy rates...")
# Nixtla automatically matches the test features contextually
predictions = nf.predict(futr_df=X_test)

# Display predictions output containing median (predict) and quantile bands
print(predictions.head())

### Pytorch Forecasting

In [ ]:
import pandas as pd
import numpy as np
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from pytorch_forecasting.data import GroupNormalizer

# =====================================================================
# 1. DATA PREPROCESSING & FORMATTING
# =====================================================================

# Load challenge data
X_train = pd.read_csv("Xtrain.csv")
Y_train = pd.read_csv("Ytrain.csv")
X_test = pd.read_csv("Xtest.csv")

# Create a master dataframe for PyTorch Forecasting data configurations
train_df = X_train.copy()
train_df['occupancy_rate'] = Y_train['target'] # Adjust string key matching your Y column

# Add a placeholder target for the test dataset to conform with dataset parsers
X_test['occupancy_rate'] = 0.0 

# Define distinct tracking groups.
# A distinct time-series sequence here is characterized by a specific Train ID + its Direction (Way)
train_df['group_id'] = train_df['train'].astype(str) + "_" + train_df['way'].astype(str)
X_test['group_id'] = X_test['train'].astype(str) + "_" + X_test['way'].astype(str)

# Generate a sequential time index column ('time_idx')
# PyTorch Forecasting strictly requires a continuous, non-gap integer tracking index.
train_df['datetime'] = pd.to_datetime(train_df['date'].astype(str) + ' ' + train_df['hour'])
X_test['datetime'] = pd.to_datetime(X_test['date'].astype(str) + ' ' + X_test['hour'])

# Generate monotonic increments grouped by time-series keys
train_df = train_df.sort_values(by=['group_id', 'datetime']).reset_index(drop=True)
train_df['time_idx'] = train_df.groupby('group_id').cumcount()

# Align test dataframe index sequences seamlessly on top of training context offsets
X_test = X_test.sort_values(by=['group_id', 'datetime']).reset_index(drop=True)
test_time_idx_offsets = train_df.groupby('group_id')['time_idx'].max() + 1

def assign_test_time_idx(row):
    # Fallback to 0 if a brand new train key combination presents itself in test set
    base_offset = test_time_idx_offsets.get(row['group_id'], 0)
    return base_offset

X_test['time_idx'] = X_test.groupby('group_id').cumcount() + X_test.apply(assign_test_time_idx, axis=1)

# Categorize and cast string features explicitly for PyTorch Embedding pipelines
categorical_cols = ['station', 'hour', 'way', 'composition']
for col in categorical_cols:
    train_df[col] = train_df[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

# Fill contextual NA gaps safely to enable tensor evaluation matrices
lag_features = ['p1q0', 'p2q0', 'p3q0', 'p0q1', 'p0q2', 'p0q3']
train_df[lag_features] = train_df[lag_features].fillna(0.0)
X_test[lag_features] = X_test[lag_features].fillna(0.0)


# =====================================================================
# 2. DEFINING THE TIMESERIESDATASET COVARIATES
# =====================================================================

max_prediction_length = 1   # Challenge target rule: Predict next station occupancy rate only
max_encoder_length = 3      # Look back window context sequence size matching deepest lag depth (p3/q3)

training_dataset = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="occupancy_rate",
    group_ids=["group_id"],
    min_encoder_length=max_encoder_length,
    max_encoder_length=max_encoder_length,
    min_prediction_length=max_prediction_length,
    max_prediction_length=max_prediction_length,
    
    # Context declarations:
    static_categoricals=["composition", "way"],
    time_varying_known_categoricals=["hour", "station"],
    time_varying_known_reals=[],
    
    # Our p and q multi-axis historical lags represent unknown future step values:
    time_varying_unknown_reals=lag_features + ["occupancy_rate"],
    
    # Scale each target within its specific series constraints boundaries safely
    target_normalizer=GroupNormalizer(groups=["group_id"], transformation="softplus"), 
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# Generate evaluation validation partitions from training sets cleanly
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, train_df, predict=True, stop_critical_features_error=False
)

# Generate target prediction input matrix frameworks from your testing datasets
testing_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset, X_test, predict=True, stop_critical_features_error=False
)

# Convert DataSets natively to standardized PyTorch DataLoader iterators
batch_size = 64
train_dataloader = training_dataset.to_dataloader(train=True, batch_size=batch_size, num_workers=2)
val_dataloader = validation_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=2)
test_dataloader = testing_dataset.to_dataloader(train=False, batch_size=batch_size * 2, num_workers=2)


# =====================================================================
# 3. INITIALIZING THE TFT MODEL & LIGHTNING TRAINER
# =====================================================================

# Build PyTorch TFT instance blueprint mappings directly from the constructed dataset schema
tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=1e-3,
    hidden_size=32,            # Networks dimensions sizing
    attention_head_size=4,     # Number of Multi-Head Self Attention units
    dropout=0.15,
    loss=QuantileLoss([0.1, 0.5, 0.9]), # Standard 10%, Median, and 90% risk band forecasting
    reduce_on_plateau_patience=4
)

# Configure PyTorch Lightning trainers with early termination constraints
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto", # Automatically triggers CUDA/MPS GPU systems if accessible
    gradient_clip_val=0.1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, min_delta=1e-4, mode="min"),
        LearningRateMonitor(logging_interval="epoch")
    ]
)

# Train network parameters
print("Fitting PyTorch Forecasting TFT on SNCF Commuter Sequences...")
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)


# =====================================================================
# 4. GENERATING INFERENCE PREDICTIONS
# =====================================================================

print("Running test matrix evaluation inference steps...")
# Retrieve outputs directly as standard PyTorch Tensors
predictions = tft.predict(test_dataloader, mode="prediction", return_x=False)

# Convert raw mathematical array structures back into a scannable Pandas DataFrame format
X_test['predicted_occupancy_median'] = predictions.detach().cpu().numpy()[:, 0]

print("\nSample Output Predictions Head Evaluation:")
print(X_test[['date', 'train', 'station', 'hour', 'predicted_occupancy_median']].head())